# <h1 style="text-align: center; font-weight: bolder;">LAB 6</h1>


# <h2 style="text-align: center; font-weight: bolder;">SETUP</h2>


* Installed the required dependencies for PyTorch training, audio I/O, visualization, perceptual/spectral losses, and CO2 tracking.
* Detected the available backend (CUDA or MPS or CPU) and set random seedsto make it global.
* Defined the main experiment params: 16 kHz sample rate and fixed 1 second clips (16,000 samples) to keep training consistent and efficient.

In [1]:
import os, sys, zipfile, random, time, pathlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import trange
from pathlib import Path
from codecarbon import EmissionsTracker

print("PWD:", pathlib.Path().resolve())
print("Root:", list(pathlib.Path("/mnt").glob("*")))

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Device:", device)

torch.set_num_threads(max(1, (os.cpu_count() or 4) - 1))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)

SAMPLE_RATE = 16000
TARGET_LENGTH = 16000
BATCH_SIZE = 64
N_EPOCHS = 50
N_EPOCHS_PERC = 50

DATA_ROOT = None

PWD: /Users/meco/Library/CloudStorage/OneDrive-Personal/Educación/UPF/Cursos/Machine Learning for Music/Labs/lab06
Root: []
Device: mps


# <h2 style="text-align: center; font-weight: bolder;">DATA LOADING WITH MAGNATAGATUNE</h2>


Reutilizamos parcialmente el codigo del lab anterior

En lab 5 ya teniamos: 
- Mirdata initialize
- Un dataset
- Segmentación



Implemented a robust loading strategy that works across environments:
* Modal Volume (if the dataset is mounted at /mnt/mtat-datalabs)
* mirdata (if the dataset is available in the installed mirdata version)
* Local fallback (searching for annotations_final.csv and clip_info_final.csv)


Summary: Built train/validation/test splits (80/10/10) and optionally capped the number of examples to control training time and cost (mainly when in need for it to execute fast)

In [2]:
# intento de carga con mirdata y fallback a versión local o modal

USE_MIRDATA = False

# detectar Modal

IS_MODAL = bool(os.environ.get("MODAL_TASK_ID") or os.environ.get("MODAL_ENVIRONMENT") or os.environ.get("MODAL_RUN_ID"))
# ruta real modal
MODAL_VOLUME_ROOT = Path("/mnt/mtat-datalabs")

def read_csv_flexible(path):
    try:
        return pd.read_csv(path, sep="\t")
    except Exception:
        return pd.read_csv(path, sep=",")

def make_splits_from_dfs(df_ann, df_info, audio_root, max_tracks=10000):
    df_ann.columns = [c.strip() for c in df_ann.columns]
    df_info.columns = [c.strip() for c in df_info.columns]

    if "clip_id" not in df_ann.columns:
        raise ValueError(f"annotations_final.csv missing 'clip_id'. Columns: {list(df_ann.columns)[:20]}")

    if "clip_id" not in df_info.columns or "mp3_path" not in df_info.columns:
        raise ValueError(f"clip_info_final.csv must contain 'clip_id' and 'mp3_path'. Columns: {list(df_info.columns)[:20]}")

    all_ids = df_ann["clip_id"].values
    if len(all_ids) > max_tracks:
        all_ids = all_ids[:max_tracks]

    all_ids = list(all_ids)
    random.shuffle(all_ids)

    n_total = len(all_ids)
    n_train = int(0.8 * n_total)
    n_val = int(0.1 * n_total)

    train_ids = all_ids[:n_train]
    val_ids = all_ids[n_train:n_train + n_val]
    test_ids = all_ids[n_train + n_val:]

    df_info_indexed = df_info.set_index("clip_id")

    def make_df(id_list):
        rows = []
        for cid in id_list:
            if cid in df_info_indexed.index:
                rows.append({"clip_id": cid, "mp3_path": df_info_indexed.loc[cid, "mp3_path"]})
        return pd.DataFrame(rows)

    train_df = make_df(train_ids)
    val_df = make_df(val_ids)
    test_df = make_df(test_ids)

    return train_df, val_df, test_df, Path(audio_root)

if IS_MODAL and (MODAL_VOLUME_ROOT / "annotations_final.csv").exists():
    data_dir = str(MODAL_VOLUME_ROOT)
    ann_path = os.path.join(data_dir, "annotations_final.csv")
    info_path = os.path.join(data_dir, "clip_info_final.csv")

    # normalizar nombres de columnas por si vienen raros

    df_ann = read_csv_flexible(ann_path)
    df_info = read_csv_flexible(info_path)

    audio_root = os.path.join(data_dir, "audio")
    train_df, val_df, test_df, DATA_ROOT = make_splits_from_dfs(df_ann, df_info, audio_root)

    print("Using MagnaTagATune from Modal Volume.")
    print("Sizes (train, val, test):", len(train_df), len(val_df), len(test_df))
    print("DATA_ROOT:", DATA_ROOT)

else:
    try:
        import mirdata

        # algunos installs de mirdata no traen magnatagatune -> si falla, cae al except

        data_home = os.environ.get("MTAT_MIRDATA_HOME", "./data/mtat")
        mtat = mirdata.initialize("magnatagatune", data_home=data_home)
        mtat.download()
        mtat.validate()

        all_tracks_dict = mtat.load_tracks()
        all_tracks = list(all_tracks_dict.values())
        random.shuffle(all_tracks)

        n_total = len(all_tracks)
        n_train = int(0.8 * n_total)
        n_val = int(0.1 * n_total)

        train_tracks = all_tracks[:n_train]
        val_tracks = all_tracks[n_train:n_train + n_val]
        test_tracks = all_tracks[n_train + n_val:]

        USE_MIRDATA = True
        print("Using MagnaTagATune via mirdata.")
        print("Sizes (train, val, test):", len(train_tracks), len(val_tracks), len(test_tracks))

    except Exception as e:
        print("mirdata not available or dataset not supported in this mirdata install. Falling back to local dataset.")
        print("Detail:", e)

        IN_COLAB = "google.colab" in sys.modules
        if IN_COLAB:
            from google.colab import drive
            drive.mount("/content/drive")
            search_roots = [Path("/content/drive/MyDrive"), Path("/content")]
        else:
            search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

        data_path = None
        for root in search_roots:
            if root.exists():
                for p in root.rglob("annotations_final.csv"):
                    data_path = p.parent
                    break
            if data_path is not None:
                break

        if data_path is None:
            raise FileNotFoundError("Could not find annotations_final.csv in known locations")

        data_dir = str(data_path)
        ann_path = os.path.join(data_dir, "annotations_final.csv")
        info_path = os.path.join(data_dir, "clip_info_final.csv")

        df_ann = read_csv_flexible(ann_path)
        df_info = read_csv_flexible(info_path)

        audio_root = os.path.join(data_dir, "audio")
        os.makedirs(audio_root, exist_ok=True)

        full_zip = os.path.join(data_dir, "mp3_full.zip")
        if os.path.exists(full_zip) and not os.path.exists(os.path.join(audio_root, "000")):
            with zipfile.ZipFile(full_zip, "r") as z:
                z.extractall(audio_root)

        train_df, val_df, test_df, DATA_ROOT = make_splits_from_dfs(df_ann, df_info, audio_root)

        print("Using local MagnaTagATune.")
        print("Sizes (train, val, test):", len(train_df), len(val_df), len(test_df))
        print("DATA_ROOT:", DATA_ROOT)

mirdata not available or dataset not supported in this mirdata install. Falling back to local dataset.
Detail: Invalid dataset magnatagatune


KeyboardInterrupt: 

Dataset para encoder

# <h2 style="text-align: center; font-weight: bolder;">AUTOENCODER BASELINE (MSE)</h2>


Autoencoder convolucional

Implemented a 1D convolutional autoencoder:
* Encoder with 3 Conv 1D layers (stride=2) to progressively downsample the waveform.
* Fixed size fully connected bottleneck of 512 (compression rate unchanged as required).
* Decoder with ConvTranspose1D layers to reconstruct the waveform back to the original length.

Trained a baseline model using MSE loss to obtain a quantitative reference and reconstruction baseline you can trust in time domain.

In [ ]:
# dataset para autoencoder (mirdata)
class MTATAutoencoderDatasetMirdata(Dataset):
    def __init__(self, tracks, split="train"):
        self.tracks = tracks
        self.split = split

    def __len__(self):
        return len(self.tracks)

    def __getitem__(self, idx):
        # carga de datos
        track = self.tracks[idx]
        audio, sr = track.audio

        # conversión a tensor
        audio = torch.tensor(audio, dtype=torch.float32)

        # conversion a mono
        if audio.ndim > 1:
            audio = audio.mean(dim=0)

        # resample
        if sr != SAMPLE_RATE:
            audio = torchaudio.functional.resample(
                audio.unsqueeze(0),
                orig_freq=sr,
                new_freq=SAMPLE_RATE
            ).squeeze(0)

        # tratamiento de audio largo (como seccion 2)
        audio = self._crop_or_pad(audio)

        return {"audio": audio}

    def _crop_or_pad(self, audio):
        length = audio.shape[-1]
        if length > TARGET_LENGTH:
            if self.split == "train":
                start = random.randint(0, length - TARGET_LENGTH)
            else:
                start = (length - TARGET_LENGTH) // 2
            audio = audio[start:start + TARGET_LENGTH]
        elif length < TARGET_LENGTH:
            pad = TARGET_LENGTH - length
            audio = F.pad(audio, (0, pad))
        return audio



# dataset para autoencoder (local: CSV + mp3)
class MTATAutoencoderDatasetLocal(Dataset):
    def __init__(self, df, split="train", data_root=None):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.data_root = Path(data_root) if data_root is not None else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # carga de datos
        row = self.df.iloc[idx]
        mp3_rel = str(row["mp3_path"])

        # normalización de ruta (evita duplicar 'audio/audio/...')
        if mp3_rel.startswith("audio/"):
            mp3_rel = mp3_rel[len("audio/"):]

        audio_path = self.data_root / mp3_rel

        # lectura de audio (evita torchcodec/ffmpeg)
        audio_np, sr = sf.read(str(audio_path), always_2d=True)  # (T, C)

        # conversión a tensor (C, T)
        audio = torch.tensor(audio_np, dtype=torch.float32).T

        # conversion a mono
        if audio.size(0) > 1:
            audio = audio.mean(dim=0, keepdim=True)

        # resample
        if sr != SAMPLE_RATE:
            audio = torchaudio.functional.resample(
                audio,
                orig_freq=sr,
                new_freq=SAMPLE_RATE
            )
            sr = SAMPLE_RATE

        audio = audio.squeeze(0)

        # tratamiento de audio largo (como seccion 2)
        audio = self._crop_or_pad(audio)

        return {"audio": audio}

    def _crop_or_pad(self, audio):
        length = audio.shape[-1]
        if length > TARGET_LENGTH:
            if self.split == "train":
                start = random.randint(0, length - TARGET_LENGTH)
            else:
                start = (length - TARGET_LENGTH) // 2
            audio = audio[start:start + TARGET_LENGTH]
        elif length < TARGET_LENGTH:
            pad = TARGET_LENGTH - length
            audio = F.pad(audio, (0, pad))
        return audio


# creación de datasets y dataloaders
if USE_MIRDATA:
    train_dataset = MTATAutoencoderDatasetMirdata(train_tracks, split="train")
    val_dataset   = MTATAutoencoderDatasetMirdata(val_tracks,   split="val")
    test_dataset  = MTATAutoencoderDatasetMirdata(test_tracks,  split="test")
else:
    if DATA_ROOT is None:
        raise ValueError("DATA_ROOT no está definido en fallback local")
    train_dataset = MTATAutoencoderDatasetLocal(train_df, split="train", data_root=DATA_ROOT)
    val_dataset   = MTATAutoencoderDatasetLocal(val_df,   split="valid", data_root=DATA_ROOT)
    test_dataset  = MTATAutoencoderDatasetLocal(test_df,  split="test",  data_root=DATA_ROOT)
import os


import platform

IS_MACOS = (platform.system().lower() == "darwin")

if IS_MACOS:
    NUM_WORKERS = 0
else:
    NUM_WORKERS = min(8, (os.cpu_count() or 8))

pin_memory = (device == "cuda")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None
)

print("DataLoaders listos:")
print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))
print("Test batches: ", len(test_loader))
print("NUM_WORKERS:", NUM_WORKERS)

t0 = time.time()
_ = next(iter(train_loader))
print("Tiempo para 1 batch:", time.time() - t0, "sec")

DataLoaders listos:
Train batches: 125
Val batches:   16
Test batches:  16
NUM_WORKERS: 0


In [ ]:
import auraloss

mse_loss = nn.MSELoss()

mrstft = auraloss.freq.MultiResolutionSTFTLoss(
    fft_sizes=[512, 1024, 2048],
    hop_sizes=[128, 256, 512],
    win_lengths=[512, 1024, 2048]
).to(device)

def perceptual_loss(x_rec, x):
    return mrstft(x_rec, x)

def combined_loss(x_rec, x, alpha=0.01):
    l_mse = mse_loss(x_rec, x)
    l_stft = perceptual_loss(x_rec, x)
    return l_mse + alpha * l_stft, l_mse, l_stft

In [ ]:
# autoencoder convolucional
class AudioAutoencoder(nn.Module):
    def __init__(self, input_length=TARGET_LENGTH, bottleneck_size=512):
        super().__init__()
        self.input_length = input_length
        self.bottleneck_size = bottleneck_size

        # encoder
        self.enc_conv1 = nn.Conv1d(1, 16, kernel_size=9, stride=2, padding=4)
        self.enc_conv2 = nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4)
        self.enc_conv3 = nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4)

        length_after = input_length
        for _ in range(3):
            length_after = int(np.ceil(length_after / 2))

        self.flatten_dim = 64 * length_after

        # bottleneck (NO cambiar compresión)
        self.fc_enc = nn.Linear(self.flatten_dim, bottleneck_size)
        self.fc_dec = nn.Linear(bottleneck_size, self.flatten_dim)

        # decoder
        self.dec_conv1 = nn.ConvTranspose1d(64, 32, kernel_size=9, stride=2, padding=4, output_padding=1)
        self.dec_conv2 = nn.ConvTranspose1d(32, 16, kernel_size=9, stride=2, padding=4, output_padding=1)
        self.dec_conv3 = nn.ConvTranspose1d(16, 1,  kernel_size=9, stride=2, padding=4, output_padding=1)

    def encode(self, x):
        x = F.relu(self.enc_conv1(x))
        x = F.relu(self.enc_conv2(x))
        x = F.relu(self.enc_conv3(x))
        x = x.flatten(start_dim=1)
        z = self.fc_enc(x)
        return z

    def decode(self, z):
        x = self.fc_dec(z)
        x = x.view(x.size(0), 64, -1)
        x = F.relu(self.dec_conv1(x))
        x = F.relu(self.dec_conv2(x))
        x = self.dec_conv3(x)
        x = x[..., :self.input_length]
        return x

    def forward(self, x):
        z = self.encode(x)
        x_rec = self.decode(z)
        return x_rec, z

# sanity check
tmp = AudioAutoencoder(input_length=TARGET_LENGTH, bottleneck_size=512).to(device)
n_params = sum(p.numel() for p in tmp.parameters() if p.requires_grad)
print("AudioAutoencoder listo. Params:", f"{n_params:,}")
del tmp

AudioAutoencoder listo. Params: 131,247,041


In [ ]:
def train_autoencoder(
    model,
    train_loader,
    val_loader,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    use_perceptual=False,
    alpha=0.01,
    project_name="lab6_autoencoder"
):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    val_mse_list = []
    val_stft_list = []

    tracker = EmissionsTracker(
        project_name=project_name,
        log_level="error",
        save_to_file=True
    )
    tracker.start()
    t0 = time.time()

    for epoch in range(n_epochs):
        model.train()
        batch_train_losses = []

        for batch in train_loader:
            x = batch["audio"].to(device, non_blocking=True).unsqueeze(1)

            optimizer.zero_grad(set_to_none=True)
            x_rec, _ = model(x)

            if use_perceptual:
                loss, _, _ = combined_loss(x_rec, x, alpha=alpha)
            else:
                loss = mse_loss(x_rec, x)

            loss.backward()
            optimizer.step()
            batch_train_losses.append(loss.item())

        train_losses.append(float(np.mean(batch_train_losses)))

        model.eval()
        batch_val_losses = []
        batch_val_mse = []
        batch_val_stft = []

        with torch.no_grad():
            for batch in val_loader:
                x = batch["audio"].to(device, non_blocking=True).unsqueeze(1)
                x_rec, _ = model(x)

                v_mse = mse_loss(x_rec, x)
                batch_val_mse.append(v_mse.item())

                if use_perceptual:
                    v_loss, _, v_stft = combined_loss(x_rec, x, alpha=alpha)
                    batch_val_losses.append(v_loss.item())
                    batch_val_stft.append(v_stft.item())
                else:
                    batch_val_losses.append(v_mse.item())

        val_losses.append(float(np.mean(batch_val_losses)))
        val_mse_list.append(float(np.mean(batch_val_mse)))

        if use_perceptual:
            val_stft_list.append(float(np.mean(batch_val_stft)))

    emissions_kg = tracker.stop()
    time_sec = time.time() - t0

    return model, {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_mse": val_mse_list,
        "val_stft": val_stft_list if use_perceptual else None,
        "emissions_kg": emissions_kg,
        "time_sec": time_sec,
        "time_min": time_sec / 60.0
    }

In [ ]:
# entrenamiento baseline MSE
model_mse = AudioAutoencoder(
    input_length=TARGET_LENGTH,
    bottleneck_size=512
).to(device)

model_mse, results_mse = train_autoencoder(
    model=model_mse,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    use_perceptual=False,
    project_name="lab6_autoencoder_mse"
)

results_mse

{'train_losses': [0.028240585550665855,
  0.025483182564377784,
  0.019950416520237924,
  0.015420402370393277,
  0.013651078432798386,
  0.012710920166224242,
  0.011730869360268115,
  0.010924974288791418,
  0.010563031684607267,
  0.010118167612701654,
  0.009919302571564913,
  0.009891541801393032,
  0.009658637050539255,
  0.009577999364584684,
  0.009694461315870284,
  0.009633752264082432,
  0.009561793398112058,
  0.009647416297346353,
  0.009486756820231677,
  0.009597882390022279,
  0.009477841507643461,
  0.00943627306446433,
  0.009480627045035363,
  0.009444203659892082,
  0.00942907366529107,
  0.009457815632224083,
  0.00937364562228322,
  0.009360890250653028,
  0.009306722853332759,
  0.009358848135918378,
  0.009351584807038307,
  0.009331478465348483,
  0.009345783092081547,
  0.009385434921830893,
  0.009305241487920285,
  0.009266763679683208,
  0.009241552699357272,
  0.009200247764587402,
  0.009249660506844521,
  0.009413785085082053,
  0.009306170728057624,
  0

In [ ]:
#N_EPOCHS = 8

In [ ]:
# entrenamiento con perdida perceptual
model_perc = AudioAutoencoder(
    input_length=TARGET_LENGTH,
    bottleneck_size=512
).to(device)

model_perc, results_perc = train_autoencoder(
    model=model_perc,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS_PERC,
    lr=1e-3,
    use_perceptual=True,
    alpha=0.01,
    project_name="lab6_autoencoder_perc"
)

results_perc

NameError: name 'AudioAutoencoder' is not defined

In [ ]:


# grafica de perdidas

plt.figure()
plt.plot(results_mse["train_losses"], label="Train (MSE)")
plt.plot(results_mse["val_mse"], label="Val MSE (baseline)")
plt.xlabel("Epoch")
plt.ylabel("Loss / MSE")
plt.legend()
plt.title("Loss curves (baseline)")
plt.grid(True)
plt.show()

if "results_perc" in globals():
    plt.figure()
    plt.plot(results_perc["train_losses"], label="Train (MSE + STFT)")
    plt.plot(results_perc["val_mse"], label="Val MSE (perceptual)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss / MSE")
    plt.legend()
    plt.title("Loss curves (comparison in Val MSE)")
    plt.grid(True)
    plt.show()
  
    # grafica extra: solo para ver STFT val del perceptual

    if results_perc["val_stft"] is not None:
        plt.figure()
        plt.plot(results_perc["val_stft"], label="Val STFT (perceptual)")
        plt.xlabel("Epoch")
        plt.ylabel("STFT loss")
        plt.legend()
        plt.title("Val STFT curve (perceptual)")
        plt.grid(True)
        plt.show()

In [ ]:
# ejemplo de reconstruccion
from IPython.display import Audio, display

batch = next(iter(test_loader))
x = batch["audio"][:4].float().to(device).unsqueeze(1)

model_mse.eval()
model_perc.eval()

with torch.no_grad():
    x_rec_mse, _ = model_mse(x)
    x_rec_perc, _ = model_perc(x)

# waveform original vs reconstruida
idx = 0
orig = x[idx].detach().cpu().squeeze().numpy()
rec_mse = x_rec_mse[idx].detach().cpu().squeeze().numpy()
rec_perc = x_rec_perc[idx].detach().cpu().squeeze().numpy()

plt.figure(figsize=(12, 6))
plt.subplot(3, 1, 1)
plt.plot(orig)
plt.title("Original")

plt.subplot(3, 1, 2)
plt.plot(rec_mse)
plt.title("Reconstrucción baseline MSE")

plt.subplot(3, 1, 3)
plt.plot(rec_perc)
plt.title("Reconstrucción MSE + STFT")
plt.tight_layout()
plt.show()



In [ ]:
from IPython.display import Audio, display
import numpy as np
import torch

def rms(x, eps=1e-8):
    return np.sqrt(np.mean(x**2) + eps)

def match_rms(y, ref, eps=1e-8):
    ry = rms(y, eps)
    rr = rms(ref, eps)
    return y * (rr / (ry + eps))

def to_audio(x_t):
    return x_t.detach().cpu().squeeze().numpy()

def show_examples(model_a, model_b, loader, n=12, seed=0):
    g = torch.Generator()
    g.manual_seed(seed)

    model_a.eval()
    model_b.eval()

    batch = next(iter(loader))
    x = batch["audio"].float()

    if x.shape[0] < n:
        n = x.shape[0]

    idxs = torch.randperm(x.shape[0], generator=g)[:n].tolist()
    x_sel = x[idxs].to(device).unsqueeze(1)

    with torch.no_grad():
        ra, _ = model_a(x_sel)
        rb, _ = model_b(x_sel)

    for i in range(n):
        orig = to_audio(x_sel[i])
        a = to_audio(ra[i])
        b = to_audio(rb[i])

        a = match_rms(a, orig)
        b = match_rms(b, orig)

        a = np.clip(a, -1.0, 1.0)
        b = np.clip(b, -1.0, 1.0)

        print(f"Example {i+1}/{n}")
        print("Original:")
        display(Audio(orig, rate=SAMPLE_RATE))
        print("Recon (MSE):")
        display(Audio(a, rate=SAMPLE_RATE))
        print("Recon (Perceptual):")
        display(Audio(b, rate=SAMPLE_RATE))
        print("-" * 60)

show_examples(model_mse, model_perc, test_loader, n=12, seed=42)

# <h2 style="text-align: center; font-weight: bolder;">LONG AUDIOS STRATEGY</h2>


Handled variable and length audio by enforcing a fixed input length (1 second):
* Training: random crop (seeking to add temporal augmentation and improve generalization).
* Validation/Test: centered crop (to ensure consistent evaluation).

This strategy is expected to allow training the model on MTAT clips with different durations while keeping batch shapes fixed.

# <h2 style="text-align: center; font-weight: bolder;">RECONSTRUCTION IMPROVEMENT (PERCEPTUAL LOSS)</h2>


* Trained a second model using a perceptual loss based on multi resolution STFT trying to combine it with MSE.
* The goal is to improve perceived spectral quality of reconstructions (timbre, detail), even if there's not a notable improvement in MSE.

# <h2 style="text-align: center; font-weight: bolder;">RESULTS AND REPORT</h2>


* Reported training and validation curves and compared both models using a common metric (Validation MSE).
* For the perceptual model, also tracked the validation perceptual/STFT loss, mainly as an auxiliary metric.
* Measured total training time and estimated CO2 emissions using CodeCarbon for resource (and) environmental reporting.

In [ ]:
# resumen final de resultados y emisiones
def print_summary(name, res):
    print(f"=== {name} ===")

    # métrica comparable entre modelos
    if "val_mse" in res and res["val_mse"] is not None:
        print(f"Best Val MSE:  {min(res['val_mse']):.6f}")

    # métrica perceptual (solo aplica si existe)
    if "val_stft" in res and res["val_stft"] is not None:
        print(f"Best Val STFT: {min(res['val_stft']):.6f}")

    print(f"Total time:  {res['time_min']:.2f} min")
    print(f"Total CO2 Emissions: {res['emissions_kg']:.6f} kg")
    print()

print_summary("Baseline MSE", results_mse)
print_summary("MSE + STFT",   results_perc)